In [1]:
# import libraries
!pip install pyod
!pip install pandas sqlalchemy pyodbc

import numpy as np
import pandas as pd
import pyodbc
import hashlib
import re

from datetime import datetime
from sqlalchemy import create_engine

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.8/219.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.3/340.3 kB 7.1 MB/s eta 0:00:00


In [2]:
"""
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17for SQL Server};"
    "SERVER=localhost,1433;"
    "DATABASE=XWPF***;"
    "Trusted_Connection=yes;"
)
print("Connected successfully")

engine = create_engine(
    "mssql+pyodbc://@localhost/XWPF***"
    "?driver=ODBC+Driver+17for+SQL+Server"
    "&trusted_connection=yes"
)

"""

'\nconn = pyodbc.connect(\n    "DRIVER={ODBC Driver 17for SQL Server};"\n    "SERVER=localhost,1433;"\n    "DATABASE=XWPF***;"\n    "Trusted_Connection=yes;"\n)\nprint("Connected successfully")\n\nengine = create_engine(\n    "mssql+pyodbc://@localhost/XWPF***"\n    "?driver=ODBC+Driver+17for+SQL+Server"\n    "&trusted_connection=yes"\n)\n\n'

### 1.0 Landing Table 

Import the data from the landing folder/zone into a landing table. All columns in this table are not defined as to ensure that data from various sources can be ingested without data type conflicts

- prevents implicit type inference
- keeps raw data "as-is"

In [3]:
db_lnd_INDIVIDUAL_TAX_RETURNS = pd.read_csv("/kaggle/input/datasets/yitianlai/individual-tax-returns/individual_tax_returns.csv", dtype=str)
#db_lnd_INDIVIDUAL_TAX_RETURNS.head(5)

### 2.0 Structure Table 

Move data from the landing table into a structured table where each column has a defined data type

In [4]:
db_struc_INDIVIDUAL_TAX_RETURNS = db_lnd_INDIVIDUAL_TAX_RETURNS.copy()

db_struc_INDIVIDUAL_TAX_RETURNS["taxpayer_id"] = db_struc_INDIVIDUAL_TAX_RETURNS["taxpayer_id"].astype("string")
db_struc_INDIVIDUAL_TAX_RETURNS["nric"] = db_struc_INDIVIDUAL_TAX_RETURNS["nric"].astype("string")
db_struc_INDIVIDUAL_TAX_RETURNS["full_name"] = db_struc_INDIVIDUAL_TAX_RETURNS["full_name"].astype("string")
db_struc_INDIVIDUAL_TAX_RETURNS["filing_status"] = db_struc_INDIVIDUAL_TAX_RETURNS["filing_status"].astype("string")
db_struc_INDIVIDUAL_TAX_RETURNS["assessment_year"] = db_struc_INDIVIDUAL_TAX_RETURNS["assessment_year"].astype("Int64")
db_struc_INDIVIDUAL_TAX_RETURNS["filing_date"] = db_struc_INDIVIDUAL_TAX_RETURNS["filing_date"].astype("string")
db_struc_INDIVIDUAL_TAX_RETURNS["annual_income_sgd"] = db_struc_INDIVIDUAL_TAX_RETURNS["annual_income_sgd"].astype("float64").round(2)
db_struc_INDIVIDUAL_TAX_RETURNS["chargeable_income_sgd"] = db_struc_INDIVIDUAL_TAX_RETURNS["chargeable_income_sgd"].astype("float64").round(2)
db_struc_INDIVIDUAL_TAX_RETURNS["tax_payable_sgd"] = db_struc_INDIVIDUAL_TAX_RETURNS["tax_payable_sgd"].astype("float64").round(2)
db_struc_INDIVIDUAL_TAX_RETURNS["tax_paid_sgd"] = db_struc_INDIVIDUAL_TAX_RETURNS["tax_paid_sgd"].astype("float64").round(2)
db_struc_INDIVIDUAL_TAX_RETURNS["total_reliefs_sgd"] = db_struc_INDIVIDUAL_TAX_RETURNS["total_reliefs_sgd"].astype("float64").round(2)
db_struc_INDIVIDUAL_TAX_RETURNS["number_of_dependents"] = db_struc_INDIVIDUAL_TAX_RETURNS["number_of_dependents"].astype("Int64")
db_struc_INDIVIDUAL_TAX_RETURNS["occupation"] = db_struc_INDIVIDUAL_TAX_RETURNS["occupation"].astype("string")
db_struc_INDIVIDUAL_TAX_RETURNS["residential_status"] = db_struc_INDIVIDUAL_TAX_RETURNS["residential_status"].astype("string")
db_struc_INDIVIDUAL_TAX_RETURNS["postal_code"] = db_struc_INDIVIDUAL_TAX_RETURNS["postal_code"].astype("string")
db_struc_INDIVIDUAL_TAX_RETURNS["housing_type"] = db_struc_INDIVIDUAL_TAX_RETURNS["housing_type"].astype("string")
db_struc_INDIVIDUAL_TAX_RETURNS["cpf_contributions_sgd"] = db_struc_INDIVIDUAL_TAX_RETURNS["cpf_contributions_sgd"].astype("float64").round(2)
db_struc_INDIVIDUAL_TAX_RETURNS["foreign_income_sgd"] = db_struc_INDIVIDUAL_TAX_RETURNS["foreign_income_sgd"].astype("float64").round(2)

#db_struc_INDIVIDUAL_TAX_RETURNS.head(2)

### 3.0 Dim & Fact Tables 

Load data from the structured table into dim and fact tables whereas hashed business key (ID hash key) and load datetime is generated


In [5]:
db_dim_INDIVIDUAL_TAX_RETURNS = db_struc_INDIVIDUAL_TAX_RETURNS[[
    "taxpayer_id",
    "nric",
    "full_name",
    "filing_status",
    "number_of_dependents",
    "occupation",
    "residential_status",
    "postal_code",
    "housing_type"
]]

db_fact_INDIVIDUAL_TAX_RETURNS = db_struc_INDIVIDUAL_TAX_RETURNS[[
    "taxpayer_id",
    "assessment_year",
    "filing_date",
    "annual_income_sgd",
    "chargeable_income_sgd",
    "tax_payable_sgd",
    "tax_paid_sgd",
    "total_reliefs_sgd",
    "cpf_contributions_sgd",
    "foreign_income_sgd"
]]

# to add hash key and load_time
def hash_key(*args):
    row_str = "_".join([str(a) for a in args])
    return hashlib.md5(row_str.encode("utf-8")).hexdigest()

db_dim_INDIVIDUAL_TAX_RETURNS["taxpayer_id_key"] = db_dim_INDIVIDUAL_TAX_RETURNS.apply(lambda row: hash_key(row["taxpayer_id"]), axis=1)
db_fact_INDIVIDUAL_TAX_RETURNS["taxpayer_id_key"] = db_fact_INDIVIDUAL_TAX_RETURNS.apply(lambda row: hash_key(row["taxpayer_id"]), axis=1)
db_dim_INDIVIDUAL_TAX_RETURNS["dim_load_dttm"] = datetime.now()
db_fact_INDIVIDUAL_TAX_RETURNS["fact_load_dttm"] = datetime.now()

#db_dim_INDIVIDUAL_TAX_RETURNS.head(3)

/tmp/ipykernel_17/351843099.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  db_dim_INDIVIDUAL_TAX_RETURNS["taxpayer_id_key"] = db_dim_INDIVIDUAL_TAX_RETURNS.apply(lambda row: hash_key(row["taxpayer_id"]), axis=1)
/tmp/ipykernel_17/351843099.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  db_fact_INDIVIDUAL_TAX_RETURNS["taxpayer_id_key"] = db_fact_INDIVIDUAL_TAX_RETURNS.apply(lambda row: hash_key(row["taxpayer_id"]), axis=1)
/tmp/ipykernel_17/351843099.py:33: SettingWithCopyWarning: 
A value is try

In [6]:
# tables are permanentlt stored in MSSQL
"""
db_fact_INDIVIDUAL_TAX_RETURNS.to_sql(
    "t_fact_INDIVIDUAL_TAX_RETURNS",
    conn,
    if_exists="append",   
    index=False
)

db_dim_INDIVIDUAL_TAX_RETURNS.to_sql(
    "t_dim_INDIVIDUAL_TAX_RETURNS",
    conn,
    if_exists="append",   
    index=False
)

"""

'\ndb_fact_INDIVIDUAL_TAX_RETURNS.to_sql(\n    "t_fact_INDIVIDUAL_TAX_RETURNS",\n    conn,\n    if_exists="append",   \n    index=False\n)\n\ndb_dim_INDIVIDUAL_TAX_RETURNS.to_sql(\n    "t_dim_INDIVIDUAL_TAX_RETURNS",\n    conn,\n    if_exists="append",   \n    index=False\n)\n\n'

### 4.0 Semantic Table

Create "stored procedures" using dim and fact tables or semantic tables to implement business rules according to user requirements


In [7]:
class BuinessLogic:

    @staticmethod
    def clean_nric(series):
        return series.where(
            series.astype("string").str.match(r"^[STFG]\d{7}[A-Z]$"),
            "INVALID"
        )

    @staticmethod
    def clean_postal(series):
        s = series.astype("string").str.strip()
        return s.where(
            s.str.match(r"^\d{6}$") & ~s.str.startswith("9"),
            "INVALID"
        )

    @staticmethod
    def clean_filing_date(series):
        return pd.to_datetime(series, dayfirst=True, errors="coerce")

    @staticmethod
    def extract_filing_year(series):
        return (
            pd.to_datetime(series, dayfirst=True, errors="coerce")
            .dt.year.astype("Int64")
            .astype("string")
            .fillna("INVALID")
        )

In [8]:
class daily_INDIVIDUAL_TAX_RETURNS:

    def __init__(self, fact_df, dim_df, target_table_name):
        self.fact_df = fact_df
        self.dim_df = dim_df
        self.target_table_name = target_table_name
        self.df = None

    def filter_by_date(self, start_dt, end_dt):
        start_dt = pd.to_datetime(start_dt)
        end_dt = pd.to_datetime(end_dt)

        self.fact_filtered = self.fact_df[
            (pd.to_datetime(self.fact_df["fact_load_dttm"]) >= start_dt) &
            (pd.to_datetime(self.fact_df["fact_load_dttm"]) <= end_dt)
        ].copy()
        return self

    def join_fact_dim(self):
        fact_cols = ["taxpayer_id_key", "fact_load_dttm", "taxpayer_id", "assessment_year",
                     "filing_date", "annual_income_sgd", "chargeable_income_sgd",
                     "tax_payable_sgd", "tax_paid_sgd", "total_reliefs_sgd",
                     "cpf_contributions_sgd", "foreign_income_sgd"]

        dim_cols = ["taxpayer_id_key", "nric", "full_name", "filing_status",
                    "number_of_dependents", "occupation", "residential_status",
                    "postal_code", "housing_type"]

        self.df = pd.merge(
            self.dim_df[dim_cols],
            self.fact_filtered[fact_cols].rename(columns={"fact_load_dttm": "reporting_date"}),
            on="taxpayer_id_key",
            how="inner"
        )
        return self

    def apply_businesslogic(self):
        df = self.df

        df["nric"] = BuinessLogic.clean_nric(df["nric"])
        df["postal_code"] = BuinessLogic.clean_postal(df["postal_code"])

        df["filing_date"] = BuinessLogic.clean_filing_date(df["filing_date"])
        df["filing_date"] = df["filing_date"].astype("string").fillna("INVALID")
        df["filing_year"] = BuinessLogic.extract_filing_year(df["filing_date"])

        df["validation_chargeable_income_sgd"] = (df["annual_income_sgd"] - df["total_reliefs_sgd"])
        df["reporting_date"] = pd.to_datetime(self.df["reporting_date"], errors="coerce").dt.date
        self.df = df
        return self

    def drop_col(self):
        df = self.df
        df = df.drop(columns=["taxpayer_id_key"])
        self.df = df
        return self

    def load(self):
        globals()[self.target_table_name] = self.df
        return self.df

Execute the "stored procedures" to load the processed data into semantic tables based on load frequency - daily, monthly, ad-hoc

In [9]:
db_sem = daily_INDIVIDUAL_TAX_RETURNS(
    db_fact_INDIVIDUAL_TAX_RETURNS,
    db_dim_INDIVIDUAL_TAX_RETURNS,
    "t_daily_INDIVIDUAL_TAX_RETURNS"
)

db_sem.filter_by_date("2026-04-01", "2026-04-30").join_fact_dim().apply_businesslogic().drop_col().load()

/tmp/ipykernel_17/2469375186.py:20: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(series, dayfirst=True, errors="coerce")
/tmp/ipykernel_17/2469375186.py:25: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(series, dayfirst=True, errors="coerce")


,nric,full_name,filing_status,number_of_dependents,occupation,residential_status,postal_code,housing_type,reporting_date,taxpayer_id,...,filing_date,annual_income_sgd,chargeable_income_sgd,tax_payable_sgd,tax_paid_sgd,total_reliefs_sgd,cpf_contributions_sgd,foreign_income_sgd,filing_year,validation_chargeable_income_sgd
0,S1234567A,Tan Wei Ming,Single,0,Software Engineer,Resident,119077,HDB 4-room,2026-04-03,SG001,...,2024-03-15,85000.0,72050.0,8205.0,8500.0,12950.0,17000.0,0.0,2024,72050.0
1,S2345678B,Lim Hui Ling,Married,2,Senior Manager,Resident,238882,Private Condo,2026-04-03,SG002,...,2024-02-28,95000.0,79100.0,10910.0,10500.0,15900.0,19000.0,0.0,2024,79100.0
2,S3456789C,Ahmad Rahman,Single,0,Network Engineer,Resident,560123,HDB 3-room,2026-04-03,SG003,...,2024-04-10,62000.0,54050.0,4405.0,4800.0,7950.0,12400.0,0.0,2024,54050.0
3,S4567890D,Chen Li Na,Married,3,Director,Resident,249657,Landed Property,2026-04-03,SG004,...,2024-01-30,118000.0,98200.0,16640.0,16200.0,19800.0,23600.0,0.0,2024,98200.0
4,S5678901E,Raj Kumar,Single,0,Data Analyst,Resident,460123,HDB 2-room,2026-04-03,SG005,...,2024-03-20,48000.0,44050.0,2405.0,2800.0,3950.0,9600.0,0.0,2024,44050.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138,S0123456M,Vikram Reddy,Single,0,Technical Lead,Resident,079765,CBD Office,2026-04-03,SG039,...,2024-02-24,91000.0,77100.0,9210.0,8800.0,13900.0,18200.0,0.0,2024,77100.0
139,S1234567N,Angela Ho,Married,1,Business Development,Resident,238987,Private Condo,2026-04-03,SG040,...,2024-01-18,69000.0,58550.0,5355.0,5800.0,10450.0,13800.0,0.0,2024,58550.0
140,G3456789O,Michael Brown,Single,0,Expatriate VP,Non-Resident,249987,Private Condo,2026-04-03,SG041,...,2024-03-16,195000.0,175200.0,41040.0,40200.0,19800.0,0.0,25000.0,2024,175200.0
141,S4567890P,Farah Binte,Married,2,Social Media Manager,Resident,730987,HDB 3-room,2026-04-03,SG042,...,2024-04-06,46000.0,42050.0,2555.0,3100.0,3950.0,9200.0,0.0,2024,42050.0


In [10]:
"""
db_sem.to_sql(
    "t_daily_INDIVIDUAL_TAX_RETURNS",
    conn,
    if_exists="append",   
    index=False
)
"""

'\ndb_sem.to_sql(\n    "t_daily_INDIVIDUAL_TAX_RETURNS",\n    conn,\n    if_exists="append",   \n    index=False\n)\n'

### 6.0 Data Quality Check

Create a data quality metrics table to store the quality result in each time when loading the data 

In [11]:
Data_Quality_Check = t_daily_INDIVIDUAL_TAX_RETURNS[["nric", "postal_code", "chargeable_income_sgd", "validation_chargeable_income_sgd"]]

Data_Quality_Check["completeness_flag"] = Data_Quality_Check["nric"].ne("INVALID").map({True: "Yes", False: "No"})
Data_Quality_Check["validity_flag"] = ((Data_Quality_Check["nric"] != "INVALID") & (Data_Quality_Check["postal_code"] != "INVALID")).map({True: "Yes", False: "No"})
Data_Quality_Check["accuracy_flag"] = (Data_Quality_Check["chargeable_income_sgd"].round(2) == Data_Quality_Check["validation_chargeable_income_sgd"].round(2)).map({True: "Yes", False: "No"})

/tmp/ipykernel_17/2054351224.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Data_Quality_Check["completeness_flag"] = Data_Quality_Check["nric"].ne("INVALID").map({True: "Yes", False: "No"})
/tmp/ipykernel_17/2054351224.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Data_Quality_Check["validity_flag"] = ((Data_Quality_Check["nric"] != "INVALID") & (Data_Quality_Check["postal_code"] != "INVALID")).map({True: "Yes", False: "No"})
/tmp/ipykernel_17/2054351224.py:5: SettingWithCopyWarning: 
A value is

In [12]:
class DataQuality:

    def __init__(self, df: pd.DataFrame, quality_check_table: str):
        self.df = df                      
        self.quality_check_table = quality_check_table
        self.agg_df = None               

    def agg_metrics(self):
        df = self.df

        # numeric flags
        df["completeness_flag_num"] = (df["completeness_flag"] == "Yes").astype(int)
        df["validity_flag_num"] = (df["validity_flag"] == "Yes").astype(int)
        df["accuracy_flag_num"] = (df["accuracy_flag"] == "Yes").astype(int)

        # aggregation
        self.agg_df = pd.DataFrame({
            "table": "t_daily_INDIVIDUAL_TAX_RETURNS",
            "domain": ["Completeness", "Validity", "Accuracy"],
            "score_pct": [
                (df["completeness_flag_num"].mean() * 100).round(2),
                (df["validity_flag_num"].mean() * 100).round(2),
                (df["accuracy_flag_num"].mean() * 100).round(2)
            ],
            "total_records": len(df),
            "passing record":[
                sum(df["completeness_flag_num"]),
                sum(df["validity_flag_num"]),
                sum(df["accuracy_flag_num"])
            ],
            "created_dttm": datetime.now()
        })
        return self

    def load(self):
        globals()[self.quality_check_table] = self.agg_df
        return self.agg_df

In [13]:
dq = DataQuality(
    Data_Quality_Check,   
    "agg_data_quality_metrics"        
)

summary_dqresult = dq.agg_metrics().load()

/tmp/ipykernel_17/2007096260.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["completeness_flag_num"] = (df["completeness_flag"] == "Yes").astype(int)
/tmp/ipykernel_17/2007096260.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["validity_flag_num"] = (df["validity_flag"] == "Yes").astype(int)
/tmp/ipykernel_17/2007096260.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats i

In [14]:
"""
summary_dqresult.to_sql(
    "dq_t_daily_INDIVIDUAL_TAX_RETURNS",
    conn,
    if_exists="append",   
    index=False
)
"""

'\nsummary_dqresult.to_sql(\n    "dq_t_daily_INDIVIDUAL_TAX_RETURNS",\n    conn,\n    if_exists="append",   \n    index=False\n)\n'

### 8.0 View Table

Synchronize semantic tables with views so that data analysts can use them for dashboards, reporting, and further analysis. This process will be done in MSSQL

In [15]:
"""
def sp_daily_INDIVIDUAL_TAX_RETURNS(start_dt: str, end_dt: str) -> pd.DataFrame:

    start_dt = pd.to_datetime(start_dt)
    end_dt = pd.to_datetime(end_dt)

    fact_filtered = db_fact_INDIVIDUAL_TAX_RETURNS[
        (pd.to_datetime(db_fact_INDIVIDUAL_TAX_RETURNS["fact_load_dttm"]) >= start_dt) &
        (pd.to_datetime(db_fact_INDIVIDUAL_TAX_RETURNS["fact_load_dttm"]) <= end_dt)
    ].copy()

    # selection of columns 
    fact_cols = ["taxpayer_id_key", "fact_load_dttm", "taxpayer_id", "assessment_year", "filing_date",  "annual_income_sgd", "chargeable_income_sgd", "tax_payable_sgd", "tax_paid_sgd", "total_reliefs_sgd", "cpf_contributions_sgd", "foreign_income_sgd"]
    dim_cols =["taxpayer_id_key", "nric", "full_name", "filing_status", "number_of_dependents", "occupation", "residential_status", "postal_code", "housing_type"]

    # merge facts and dim tables
    t_daily_INDIVIDUAL_TAX_RETURNS = pd.merge(
        db_dim_INDIVIDUAL_TAX_RETURNS[dim_cols],
        db_fact_INDIVIDUAL_TAX_RETURNS[fact_cols].rename(columns={"fact_load_dttm":"reporting_date"}),
        on="taxpayer_id_key", how="inner"
    )

    # business logic
    #t_daily_INDIVIDUAL_TAX_RETURNS["reporting_dt"] = pd.to_datetime(t_daily_INDIVIDUAL_TAX_RETURNS["reporting_dt"]).dt.date
    t_daily_INDIVIDUAL_TAX_RETURNS["nric"] = t_daily_INDIVIDUAL_TAX_RETURNS["nric"].where(t_daily_INDIVIDUAL_TAX_RETURNS["nric"].astype("string").str.match(r"^[STFG]\d{7}[A-Z]$"), "INVALID")
    t_daily_INDIVIDUAL_TAX_RETURNS["postal_code"] = t_daily_INDIVIDUAL_TAX_RETURNS["postal_code"].astype("string").str.strip().where(lambda x:x.str.match(r"^\d{6}$") & ~x.str.startswith("9"), "INVALID")
    t_daily_INDIVIDUAL_TAX_RETURNS["filing_date"] = pd.to_datetime(t_daily_INDIVIDUAL_TAX_RETURNS["filing_date"], dayfirst=True, errors="coerce")
    t_daily_INDIVIDUAL_TAX_RETURNS["filing_date"] = t_daily_INDIVIDUAL_TAX_RETURNS["filing_date"].astype("string").fillna("INVALID")
    t_daily_INDIVIDUAL_TAX_RETURNS["filing_year"] = pd.to_datetime(t_daily_INDIVIDUAL_TAX_RETURNS["filing_date"], dayfirst=True, errors="coerce").dt.year.astype("Int64").astype("string").fillna("INVALID")
    t_daily_INDIVIDUAL_TAX_RETURNS["validation_chargeable_income_sgd"] = t_daily_INDIVIDUAL_TAX_RETURNS["annual_income_sgd"] - t_daily_INDIVIDUAL_TAX_RETURNS["total_reliefs_sgd"]

    return t_daily_INDIVIDUAL_TAX_RETURNS
"""

<>:25: SyntaxWarning: invalid escape sequence '\d'
<>:25: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_17/2260711341.py:25: SyntaxWarning: invalid escape sequence '\d'
  t_daily_INDIVIDUAL_TAX_RETURNS["nric"] = t_daily_INDIVIDUAL_TAX_RETURNS["nric"].where(t_daily_INDIVIDUAL_TAX_RETURNS["nric"].astype("string").str.match(r"^[STFG]\d{7}[A-Z]$"), "INVALID")


'\ndef sp_daily_INDIVIDUAL_TAX_RETURNS(start_dt: str, end_dt: str) -> pd.DataFrame:\n\n    start_dt = pd.to_datetime(start_dt)\n    end_dt = pd.to_datetime(end_dt)\n\n    fact_filtered = db_fact_INDIVIDUAL_TAX_RETURNS[\n        (pd.to_datetime(db_fact_INDIVIDUAL_TAX_RETURNS["fact_load_dttm"]) >= start_dt) &\n        (pd.to_datetime(db_fact_INDIVIDUAL_TAX_RETURNS["fact_load_dttm"]) <= end_dt)\n    ].copy()\n\n    # selection of columns \n    fact_cols = ["taxpayer_id_key", "fact_load_dttm", "taxpayer_id", "assessment_year", "filing_date",  "annual_income_sgd", "chargeable_income_sgd", "tax_payable_sgd", "tax_paid_sgd", "total_reliefs_sgd", "cpf_contributions_sgd", "foreign_income_sgd"]\n    dim_cols =["taxpayer_id_key", "nric", "full_name", "filing_status", "number_of_dependents", "occupation", "residential_status", "postal_code", "housing_type"]\n\n    # merge facts and dim tables\n    t_daily_INDIVIDUAL_TAX_RETURNS = pd.merge(\n        db_dim_INDIVIDUAL_TAX_RETURNS[dim_cols],\n       